In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/krupalpatel07/hsbc-bank-uks-largest-bank/hsbc.csv


In [2]:
# =====================================================
# 1. IMPORT LIBRARIES
# =====================================================

import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px

plt.style.use('default')

In [3]:
import plotly.io as pio

pio.renderers.default = 'iframe'

In [4]:
# =====================================================
# 2. LOAD DATA
# =====================================================
file_path = "/kaggle/input/datasets/krupalpatel07/hsbc-bank-uks-largest-bank/hsbc.csv"
df = pd.read_csv(file_path)

In [5]:
# =====================================================
# 3. PREPROCESSING
# =====================================================
df.columns = [c.lower() for c in df.columns]
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date')
df.set_index('date', inplace=True)

In [6]:
# =====================================================
# 4. FRACTAL HEADER
# =====================================================
from IPython.display import display, HTML

def fractal_header(text):
    display(HTML(f"""
    <div style="
        background: linear-gradient(90deg, #ff00cc, #3333ff);
        padding: 20px; border-radius: 20px; margin-top:20px;
        backdrop-filter: blur(10px);
    ">
        <h1 style="color:white; text-align:center;">{text}</h1>
    </div>
    """))

fractal_header("📊 Price as a Fractal Structure")

In [7]:
# =====================================================
# 5. PRICE VISUAL
# =====================================================
fig = px.line(df, y='close', title='HSBC Price')
fig.show()

In [8]:
# =====================================================
# 6. HURST EXPONENT FUNCTION
# =====================================================
fractal_header("🧠 Hurst Exponent (Market Memory)")

def hurst(ts):
    lags = range(2, 20)
    tau = [np.sqrt(np.std(np.subtract(ts[lag:], ts[:-lag]))) for lag in lags]
    poly = np.polyfit(np.log(lags), np.log(tau), 1)
    return poly[0]*2.0

# rolling hurst
hurst_vals = []
window = 100
for i in range(window, len(df)):
    hurst_vals.append(hurst(df['close'].values[i-window:i]))

hurst_series = pd.Series(hurst_vals, index=df.index[window:])

fig = px.line(hurst_series, title='Hurst Exponent Over Time')
fig.show()


In [9]:
# =====================================================
# 7. FRACTAL REGIME CLASSIFICATION
# =====================================================
fractal_header("⚡ Fractal Regimes")

regime = np.where(hurst_series > 0.5, 'Trending', 'Mean-Reverting')

fig = px.scatter(x=hurst_series.index, y=hurst_series, color=regime)
fig.show()

In [10]:
# =====================================================
# 8. MULTI-SCALE RETURNS
# =====================================================
fractal_header("🔍 Multi-Scale Alpha")

for w in [5, 20, 60]:
    df[f'ret_{w}'] = df['close'].pct_change(w)

fig = go.Figure()
for w in [5,20,60]:
    fig.add_trace(go.Scatter(x=df.index, y=df[f'ret_{w}'], name=f'{w}D Return'))
fig.show()

In [12]:
# =====================================================
# 10. CHAOS / VOLATILITY BURSTS
# =====================================================
fractal_header("🔥 Chaos Zones")

returns = df['close'].pct_change()
z = (returns - returns.mean())/returns.std()
df['chaos'] = np.abs(z) > 2

fig = px.scatter(df, x=df.index, y='close', color='chaos')
fig.show()

In [11]:
# =====================================================
# 9. SIGNAL ENGINE
# =====================================================
fractal_header("🎯 Fractal Signal Engine")

# align hurst
aligned = hurst_series.reindex(df.index).fillna(method='bfill')

signal = (aligned > 0.5) & (df['ret_20'] > 0)
df['signal'] = signal.astype(int)

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=df['close'], name='Price'))
fig.add_trace(go.Scatter(x=df.index[df['signal']==1], y=df['close'][df['signal']==1],
                         mode='markers', name='Signal'))
fig.show()

/tmp/ipykernel_55/1700212919.py:7: FutureWarning:

Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.



In [13]:
# =====================================================
# 11. FINAL INSIGHTS
# =====================================================
fractal_header("📌 Fractal Insights")

print("""
1. Hurst exponent reveals market memory.
2. >0.5 indicates trending behavior.
3. <0.5 suggests mean reversion.
4. Multi-scale returns uncover hidden alpha.
5. Fractal thinking improves regime-based trading.
""")

# =====================================================
# END
# =====================================================



1. Hurst exponent reveals market memory.
2. >0.5 indicates trending behavior.
3. <0.5 suggests mean reversion.
4. Multi-scale returns uncover hidden alpha.
5. Fractal thinking improves regime-based trading.

